# Analyzer
Base reproducible para estudiar la evidencia de seguridad generada por el Miner.

**Pregunta general:** ¿Cómo se distribuyen los hallazgos de seguridad en la organización y qué patrones permiten identificar prioridades de revisión?

Los hallazgos son alertas o coincidencias, no vulnerabilidades confirmadas. Los análisis fallidos no representan ausencia de problemas.

In [1]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import display


In [2]:
# Localizar el proyecto desde la carpeta principal o desde notebooks.
current_directory = Path.cwd()
project_candidates = [
    current_directory,
    current_directory.parent,
    current_directory / 'analyzer',
]
ANALYZER_ROOT = next(
    (path for path in project_candidates if (path / 'notebooks/analyzer.ipynb').is_file()),
    None,
)
if ANALYZER_ROOT is None:
    raise FileNotFoundError('No se encontró la carpeta analyzer.')

# Copia fija de la evidencia; el origen está registrado en provenance.json.
DATASET_PATH = ANALYZER_ROOT / 'data' / 'pallets' / 'dataset.json'
print(f'Dataset: {DATASET_PATH}')


Dataset: c:\Users\jamar\ciberseguridad\vulnerabilities\analyzer\data\pallets\dataset.json


In [3]:
with DATASET_PATH.open(encoding='utf-8') as dataset_file:
    dataset = json.load(dataset_file)

if not isinstance(dataset, dict):
    raise ValueError('El dataset debe ser un objeto JSON.')

supported_versions = {'1.0', '1.1'}
if dataset.get('schema_version') not in supported_versions:
    raise ValueError('Versión de dataset no compatible.')

for section in ('repositories', 'findings'):
    records = dataset.get(section)
    if not isinstance(records, list):
        raise ValueError(f'La sección {section} debe ser una lista.')
    if any(not isinstance(record, dict) for record in records):
        raise ValueError(f'La sección {section} contiene registros inválidos.')

repository_records = dataset['repositories']
finding_records = dataset['findings']
repository_names = {record['repository'] for record in repository_records}

for finding in finding_records:
    if finding.get('repository') not in repository_names:
        raise ValueError('Hay hallazgos de repositorios ausentes del dataset.')
    for field in ('tool', 'vulnerability_id'):
        if not isinstance(finding.get(field), str) or not finding[field].strip():
            raise ValueError(f'Un hallazgo no contiene un {field} válido.')

print(f"Organización: {dataset['organization']}")
print(f'Repositorios: {len(repository_records)}')
print(f'Hallazgos: {len(finding_records)}')



Organización: pallets
Repositorios: 17
Hallazgos: 310


In [4]:
repository_columns = [
    'repository',
    'clone_status',
    'codeql_status',
    'grype_status',
    'clone_error',
    'codeql_error',
    'grype_error',
]
finding_columns = [
    'repository',
    'tool',
    'vulnerability_id',
    'description',
    'severity',
    'severity_kind',
    'locations',
    'language',
    'package',
    'version',
    'commit',
    'scores',
]
score_columns = [
    'finding_index',
    'repository',
    'tool',
    'value',
    'system',
    'source',
    'version',
    'vector',
    'vulnerability_id',
]

repositories = pd.DataFrame(repository_records).reindex(columns=repository_columns)
findings = pd.DataFrame(finding_records).reindex(columns=finding_columns)

score_rows = []
for finding_index, finding in enumerate(finding_records):
    for score in finding.get('scores', []):
        score_rows.append({
            **score,
            'finding_index': finding_index,
            'repository': finding['repository'],
            'tool': finding['tool'],
        })
scores = pd.DataFrame(score_rows).reindex(columns=score_columns)

display(repositories.head())
display(findings.head())
display(scores.head())



,repository,clone_status,codeql_status,grype_status,clone_error,codeql_error,grype_error
0,pallets/.github,cloned,database_failed,analyzed,None,CodeQL no pudo crear las bases de datos,None
1,pallets/actions,cloned,database_failed,analyzed,None,CodeQL no pudo crear las bases de datos,None
2,pallets/click,cloned,analyzed,analyzed,None,None,None
3,pallets/flask,cloned,analyzed,analyzed,None,None,None
4,pallets/flask-docs,cloned,analyzed,analyzed,None,None,None


,repository,tool,vulnerability_id,description,severity,severity_kind,locations,language,package,version,commit,scores
0,pallets/click,codeql,py/clear-text-logging-sensitive-data,This expression logs [sensitive data (password...,error,sarif_level,[src/click/utils.py],python,None,None,None,"[{'value': 7.5, 'system': 'security-severity',..."
1,pallets/click,grype,GHSA-4gg8-gxpx-9rph,uv is vulnerable to arbitrary file write throu...,Medium,vulnerability_severity,[\uv.lock],None,uv,0.11.3,06b2a678741131fd577ce170e23e5ca0aeba0309,[]
2,pallets/click,grype,GHSA-5p39-cfhj-2xmp,AnyIO process-pool workers can block indefinit...,Medium,vulnerability_severity,[\uv.lock],None,anyio,4.13.0,06b2a678741131fd577ce170e23e5ca0aeba0309,"[{'value': 6.8, 'system': 'CVSS', 'source': 'h..."
3,pallets/click,grype,GHSA-65pc-fj4g-8rjx,Internationalized Domain Names in Applications...,Medium,vulnerability_severity,[\uv.lock],None,idna,3.11,06b2a678741131fd577ce170e23e5ca0aeba0309,"[{'value': 5.3, 'system': 'CVSS', 'source': 'h..."
4,pallets/click,grype,GHSA-6w46-j5rx-g56g,pytest has vulnerable tmpdir handling,Medium,vulnerability_severity,[\uv.lock],None,pytest,9.0.2,06b2a678741131fd577ce170e23e5ca0aeba0309,"[{'value': 6.8, 'system': 'CVSS', 'source': 'h..."


,finding_index,repository,tool,value,system,source,version,vector,vulnerability_id
0,0,pallets/click,codeql,7.5,security-severity,codeql,None,None,py/clear-text-logging-sensitive-data
1,2,pallets/click,grype,6.8,CVSS,https://github.com/advisories/GHSA-5p39-cfhj-2xmp,4.0,CVSS:4.0/AV:L/AC:L/AT:N/PR:L/UI:N/VC:N/VI:N/VA...,GHSA-5p39-cfhj-2xmp
2,2,pallets/click,grype,6.8,CVSS,security-advisories@github.com,4.0,CVSS:4.0/AV:L/AC:L/AT:N/PR:L/UI:N/VC:N/VI:N/VA...,CVE-2026-64847
3,3,pallets/click,grype,5.3,CVSS,https://github.com/advisories/GHSA-65pc-fj4g-8rjx,3.1,CVSS:3.1/AV:N/AC:L/PR:N/UI:N/S:U/C:N/I:N/A:L,GHSA-65pc-fj4g-8rjx
4,3,pallets/click,grype,6.9,CVSS,https://github.com/advisories/GHSA-65pc-fj4g-8rjx,4.0,CVSS:4.0/AV:N/AC:L/AT:N/PR:N/UI:N/VC:N/VI:N/VA...,GHSA-65pc-fj4g-8rjx


## Cobertura antes del análisis
Separar los estados por herramienta. `not_run`, fallos y análisis parciales deben considerarse al definir las poblaciones de comparación.

In [5]:
status_columns = ['clone_status', 'codeql_status', 'grype_status']
repository_stages = repositories.melt(
    id_vars='repository',
    value_vars=status_columns,
    var_name='stage',
    value_name='status',
)
coverage = (
    repository_stages.groupby(['stage', 'status'])
    .size()
    .rename('repositories')
    .reset_index()
)
findings_by_tool = (
    findings.groupby('tool')
    .size()
    .rename('findings')
    .reset_index()
)

display(coverage)
display(findings_by_tool)



,stage,status,repositories
0,clone_status,cloned,17
1,codeql_status,analyzed,13
2,codeql_status,database_failed,4
3,grype_status,analyzed,17


,tool,findings
0,codeql,86
1,grype,224


## Hallazgos repetidos por repositorio

Agrupamos por herramienta, identificador y repositorio. Coincidencias totales mide registros del Miner; repositorios afectados mide la extensión del hallazgo entre proyectos. Las reglas de CodeQL y los identificadores de Grype se mantienen separados. No se agrupa por puntaje ni se eliminan coincidencias de paquetes distintos.

In [6]:
frecuencias = (
    findings.groupby(['tool', 'vulnerability_id', 'repository'])
    .size()
    .rename('coincidencias')
    .reset_index()
    .sort_values(['tool', 'vulnerability_id', 'repository'])
)

repetidos = (
    frecuencias.groupby(['tool', 'vulnerability_id'])
    .agg(
        repositorios_afectados=('repository', 'nunique'),
        coincidencias_totales=('coincidencias', 'sum'),
    )
    .reset_index()
    .sort_values(
        ['repositorios_afectados', 'coincidencias_totales', 'tool', 'vulnerability_id'],
        ascending=[False, False, True, True],
    )
    .reset_index(drop=True)
)

display(frecuencias, repetidos)

# Matriz de coincidencias: filas por hallazgo y columnas por repositorio.
matriz_hallazgos = frecuencias.pivot(
    index=['tool', 'vulnerability_id'], columns='repository', values='coincidencias'
).reindex(columns=sorted(repositories['repository'].unique())).fillna(0).astype(int)
display(matriz_hallazgos)
# Un cero solo indica ausencia de registros; consultar coverage para saber si hubo análisis.


,tool,vulnerability_id,repository,coincidencias
0,codeql,js/functionality-from-untrusted-source,pallets/flask-website,2
1,codeql,py/clear-text-logging-sensitive-data,pallets/click,1
2,codeql,py/flask-debug,pallets/flask-website,1
3,codeql,py/incomplete-url-substring-sanitization,pallets/werkzeug,1
4,codeql,py/insecure-protocol,pallets/werkzeug,1
...,...,...,...,...
215,grype,GHSA-x78j-v8h9-3j2q,pallets/markupsafe,1
216,grype,GHSA-x78j-v8h9-3j2q,pallets/pallets-sphinx-themes,1
217,grype,GHSA-x78j-v8h9-3j2q,pallets/quart,1
218,grype,GHSA-x78j-v8h9-3j2q,pallets/website,1


,tool,vulnerability_id,repositorios_afectados,coincidencias_totales
0,grype,GHSA-94p9-xgh2-xp45,8,8
1,grype,GHSA-9h9j-4vrj-gf7g,8,8
2,grype,GHSA-p58f-9548-mpm2,8,8
3,grype,GHSA-x78j-v8h9-3j2q,8,8
4,grype,GHSA-8988-9cw3-xx77,7,7
...,...,...,...,...
92,grype,GHSA-q34m-jh98-gwm2,1,1
93,grype,GHSA-q58r-hwc8-rm9j,1,1
94,grype,GHSA-qj8w-gfj5-8c6v,1,1
95,grype,GHSA-vxmc-5x29-h64v,1,1


repository                                       pallets/.github  \
tool   vulnerability_id                                            
codeql js/functionality-from-untrusted-source                  0   
       py/clear-text-logging-sensitive-data                    0   
       py/flask-debug                                          0   
       py/incomplete-url-substring-sanitization                0   
       py/insecure-protocol                                    0   
...                                                          ...   
grype  GHSA-w853-jp5j-5j7f                                     0   
       GHSA-wqp7-x3pw-xc5r                                     0   
       GHSA-x746-7m8f-x49c                                     0   
       GHSA-x78j-v8h9-3j2q                                     0   
       GHSA-xwg4-73v4-xw9w                                     0   

repository                                       pallets/actions  \
tool   vulnerability_id                                            
codeql js/functionality-from-untrusted-source                  0   
       py/clear-text-logging-sensitive-data                    0   
       py/flask-debug                                          0   
       py/incomplete-url-substring-sanitization                0   
       py/insecure-protocol                                    0   
...                                                          ...   
grype  GHSA-w853-jp5j-5j7f                                     0   
       GHSA-wqp7-x3pw-xc5r                                     0   
       GHSA-x746-7m8f-x49c                                     0   
       GHSA-x78j-v8h9-3j2q                                     0   
       GHSA-xwg4-73v4-xw9w                                     0   

repository                                       pallets/click  pallets/flask  \
tool   vulnerability_id                                                         
codeql js/functionality-from-untrusted-source                0              0   
       py/clear-text-logging-sensitive-data                  1              0   
       py/flask-debug                                        0              0   
       py/incomplete-url-substring-sanitization              0              0   
       py/insecure-protocol                                  0              0   
...                                                        ...            ...   
grype  GHSA-w853-jp5j-5j7f                                   0              0   
       GHSA-wqp7-x3pw-xc5r                                   1              0   
       GHSA-x746-7m8f-x49c                                   1              0   
       GHSA-x78j-v8h9-3j2q                                   1              1   
       GHSA-xwg4-73v4-xw9w                                   0              0   

repository                                       pallets/flask-docs  \
tool   vulnerability_id                                               
codeql js/functionality-from-untrusted-source                     0   
       py/clear-text-logging-sensitive-data                       0   
       py/flask-debug                                             0   
       py/incomplete-url-substring-sanitization                   0   
       py/insecure-protocol                                       0   
...                                                             ...   
grype  GHSA-w853-jp5j-5j7f                                        0   
       GHSA-wqp7-x3pw-xc5r                                        0   
       GHSA-x746-7m8f-x49c                                        0   
       GHSA-x78j-v8h9-3j2q                                        0   
       GHSA-xwg4-73v4-xw9w                                        0   

repository                                       pallets/flask-ext-migrate  \
tool   vulnerability_id                                                      
codeql js/functionality-from-untrusted-source                            0   
       py/clear-te

## Preguntas de investigación
1. ¿Los hallazgos se concentran en pocos repositorios?
2. ¿Qué reglas de CodeQL se repiten entre proyectos?
3. ¿Dónde se concentran los puntajes altos, distinguiendo sus escalas y fuentes?
4. ¿Qué paquetes y vulnerabilidades de Grype comparten varios repositorios?
5. ¿Existe relación entre los conteos de CodeQL y Grype entre repositorios con cobertura completa?

Para cada pregunta: método, resultado, interpretación y limitaciones. No mezclar CVSS y security-severity ni contar cada puntaje como un hallazgo nuevo. Los conteos no están ajustados por tamaño del código o número total de dependencias.

## Resultados para el Visualizer
Exportar aquí, al desarrollar los análisis, las métricas y tablas derivadas junto con sus definiciones, filtros y observaciones. Mantener la evidencia original en el Miner.